# OSL End-to-End Notebook (DRQN / DQN)

Same 2D OSL stack as `PPO_framework.ipynb`, but with the unified
recurrent / feed-forward Q-net agent (DRQN when `recurrent=True`,
DQN when `recurrent=False`).

- Env: `StaticEnv` / `DynamicEnv` (2D Gaussian / turbulent plume)
- Agent: `DRQNAgent` (discrete action set `{RUN, CAST, TURN_L, TURN_R}`
  mapped to the env's `Box([v, omega, cast])` action space).


In [ ]:
%pip uninstall -y gym
%pip install gymnasium==0.29.1 "numpy<2.0" matplotlib imageio imageio[ffmpeg] scipy


In [ ]:
import os
import io
from collections import deque

import numpy as np
import torch
import matplotlib.pyplot as plt
import imageio
from IPython.display import Image as DisplayImage, display

# Repo modules — reuse the same env / network / agent code as the .py CLI.
from src.envs.osl_env_2d import StaticEnv, DynamicEnv
from src.agents.drqn_agent import DRQNAgent, A_RUN, A_CAST, A_TURN_L, A_TURN_R, N_ACTIONS
from src.utils.buffer import EpisodeReplayBuffer

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

os.makedirs("checkpoints_drqn", exist_ok=True)


## 1. Train

4-phase curriculum mirroring `PPO_framework.ipynb`: static → coef 0.3 → 0.6 → 1.0.
DRQN is single-env (no SubprocVecEnv), so totals are scaled down vs PPO.


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PHASES = [
    ("static",       20_000),
    ("dynamic_0.3",  10_000),
    ("dynamic_0.6",  10_000),
    ("dynamic_1.0",  20_000),
]

def make_env(env_kind):
    if env_kind == "static":
        return StaticEnv()
    coef = float(env_kind.split("_")[1])
    return DynamicEnv(noise_coef=coef)

def linear_eps(ep, start=1.0, end=0.05, decay_eps=4000):
    frac = min(1.0, ep / max(1, decay_eps))
    return start + frac * (end - start)

env = make_env(PHASES[0][0])
agent = DRQNAgent(
    obs_dim=env.observation_space.shape[0],
    action_low=env.action_space.low,
    action_high=env.action_space.high,
    device=DEVICE,
    rnn_hidden=147,
    lr=1e-4,
    gamma=0.99,
    recurrent=True,   # set False for vanilla DQN
)
buffer = EpisodeReplayBuffer(cap_steps=150_000)

ep_returns, success_window = [], deque(maxlen=100)
global_ep = 0

for phase_kind, phase_episodes in PHASES:
    env = make_env(phase_kind)
    print(f"=== Phase {phase_kind}  ({phase_episodes} eps) ===")
    for _ in range(phase_episodes):
        global_ep += 1
        obs, _ = env.reset(seed=42 + global_ep)
        h = None
        traj = []
        ep_ret, success = 0.0, False
        while True:
            eps = linear_eps(global_ep)
            a_idx, h = agent.get_action(obs, h, epsilon=eps)
            env_action = agent.to_env_action(a_idx)
            next_obs, r, terminated, truncated, info = env.step(env_action)
            traj.append((obs, a_idx, float(r), next_obs, float(terminated)))
            obs = next_obs
            ep_ret += float(r)
            if info.get("is_success"):
                success = True
            if terminated or truncated:
                break
        buffer.add_episode(traj)
        ep_returns.append(ep_ret)
        success_window.append(float(success))

        if len(buffer) >= 5000:
            agent.update(buffer.sample(batch_size=128, seq_len=16))
            if global_ep % 20 == 0:
                agent.sync_target()

        if global_ep % 200 == 0:
            print(f"  ep {global_ep:5d}  ret={ep_ret:7.2f}  succ100={np.mean(success_window):.3f}  eps={eps:.3f}")

    agent.save(f"checkpoints_drqn/{phase_kind}.pt")
    print(f"  saved checkpoints_drqn/{phase_kind}.pt")


## 2. Evaluate + Elite Seed Search

Same idea as `find_elite_seeds` / `render_elite_gif` in PPO_framework, but for
the deterministic DRQN policy with the discrete action adapter.


In [ ]:
def find_elite_seeds(agent, env_kind="dynamic_1.0", n_to_find=3, min_casts=1, max_casts=300):
    env = make_env(env_kind)
    elites = []
    for trial in range(500):
        seed = 20000 + trial
        obs, _ = env.reset(seed=seed)
        h = None
        casts = 0
        success = False
        for _ in range(300):
            a_idx, h = agent.get_action_deterministic(obs, h)
            if a_idx == A_CAST:
                casts += 1
            obs, _, term, trunc, info = env.step(agent.to_env_action(a_idx))
            if info.get("is_success"):
                success = True
            if term or trunc:
                break
        if success and min_casts <= casts <= max_casts:
            elites.append(seed)
            print(f"[seed {seed}] success, casts={casts}")
        if len(elites) >= n_to_find:
            break
    return elites

def render_elite_gif(agent, seed, env_kind="dynamic_1.0", filename="drqn_turbulent.gif"):
    env = make_env(env_kind)
    obs, _ = env.reset(seed=seed)
    h = None
    frames, traj_x, traj_y, cx, cy = [], [], [], [], []
    for t in range(300):
        a_idx, h = agent.get_action_deterministic(obs, h)
        traj_x.append(env.x); traj_y.append(env.y)
        if a_idx == A_CAST or getattr(env, "in_cast", False):
            cx.append(env.x); cy.append(env.y)

        field = getattr(env, "_field_view", None)
        if field is None:
            res = 100
            xs = np.linspace(-env.L, env.L, res); ys = np.linspace(-env.L, env.L, res)
            X, Y = np.meshgrid(xs, ys)
            field = np.clip(np.asarray(env._conc(X, Y), dtype=np.float32), 0.0, 1.0)

        fig, ax = plt.subplots(figsize=(7, 7))
        fig.patch.set_facecolor("black"); ax.set_facecolor("black")
        ax.imshow(field, extent=[-env.L, env.L, -env.L, env.L], origin="lower", cmap="inferno", vmin=0, vmax=1)
        ax.plot(traj_x, traj_y, color="#50dcff", linewidth=2.0, alpha=0.8)
        if cx:
            ax.scatter(cx, cy, color="white", marker="*", s=150, edgecolors="black", zorder=10)
        ax.scatter(env.src_x, env.src_y, color="lime", marker="P", s=150, zorder=11)
        circle = plt.Circle((env.src_x, env.src_y), env.r_goal, color="gray", fill=False)
        ax.add_patch(circle)
        ax.set_title(f"DRQN {env_kind} seed={seed} step={t}", color="white")
        ax.tick_params(colors="white")

        buf = io.BytesIO()
        plt.savefig(buf, format="png", dpi=80, bbox_inches="tight", facecolor="black")
        plt.close(fig); buf.seek(0)
        frames.append(np.array(imageio.v2.imread(buf)))

        obs, _, term, trunc, _ = env.step(agent.to_env_action(a_idx))
        if term or trunc:
            break

    imageio.mimsave(filename, frames, fps=15)
    display(DisplayImage(data=open(filename, "rb").read(), format="png"))

elites = find_elite_seeds(agent)
if elites:
    render_elite_gif(agent, elites[0], filename="drqn_turbulent.gif")
else:
    print("no elite seed found")
